# Episode Clips Viewer

Browse annotated episode MP4 clips produced by `scripts/render_episode_clips.py`.  
Select a date and clip from the dropdowns — the video plays inline.

In [ ]:
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

REPO_ROOT = Path("..").resolve()
OUTPUT_ROOT = REPO_ROOT / "output"

def discover_dates():
    """Return sorted list of dates that have a clips/ subdirectory with ≥1 MP4."""
    dates = []
    if not OUTPUT_ROOT.exists():
        return dates
    for d in sorted(OUTPUT_ROOT.iterdir()):
        clips_dir = d / "clips"
        if clips_dir.is_dir() and list(clips_dir.glob("*.mp4")):
            dates.append(d.name)
    return dates

def discover_clips(date_str):
    clips_dir = OUTPUT_ROOT / date_str / "clips"
    return sorted(clips_dir.glob("*.mp4"))

def load_episode_meta(date_str, clip_path):
    """Try to find episode metadata from episodes.jsonl for this clip."""
    ep_path = OUTPUT_ROOT / date_str / "episodes.jsonl"
    if not ep_path.exists():
        return None
    # Extract callsign from filename: NN_CALLSIGN_HHMMSSZ.mp4
    stem = clip_path.stem  # e.g. "01_EIN108_042709Z"
    parts = stem.split("_", 2)
    if len(parts) < 2:
        return None
    callsign = parts[1]
    episodes = [json.loads(l) for l in ep_path.read_text().splitlines() if l.strip()]
    # Find the matching episode by callsign + approximate onset time
    time_part = parts[2] if len(parts) > 2 else ""
    for ep in episodes:
        if ep["callsign"] == callsign:
            onset = ep["onset"]  # e.g. "2026-04-08T04:27:09+00:00"
            onset_tag = onset[11:19].replace(":", "") + "Z"  # "042709Z"
            if onset_tag == time_part:
                return ep
    return None

dates = discover_dates()
print("Dates with clips:", dates)

In [ ]:
if not dates:
    print("No clips found. Run: uv run python scripts/render_episode_clips.py --date YYYY-MM-DD")
else:
    date_dd = widgets.Dropdown(
        options=dates,
        description="Date:",
        layout=widgets.Layout(width="200px"),
    )
    clip_dd = widgets.Dropdown(
        options=[],
        description="Episode:",
        layout=widgets.Layout(width="420px"),
    )
    btn_prev = widgets.Button(description="◀", layout=widgets.Layout(width="48px"))
    btn_next = widgets.Button(description="▶", layout=widgets.Layout(width="48px"))
    out = widgets.Output()

    def _show_clip(_=None):
        clip_path = clip_dd.value
        if clip_path is None:
            return
        date_str = date_dd.value
        ep = load_episode_meta(date_str, clip_path)

        # Path relative to notebook server root (notebooks/ is one level below repo root)
        rel = "../" + str(clip_path.relative_to(REPO_ROOT))

        # Build metadata table
        if ep:
            onset_utc = ep["onset"].replace("+00:00", " UTC")
            end_utc   = ep["end"].replace("+00:00", " UTC")
            score_col = "#4f4" if ep["peak_score"] >= 0.5 else "#fa4"
            length_str = f"{ep['peak_contrail_length_m']:.0f} m" if ep.get("peak_contrail_length_m") else "—"
            meta_html = f"""
            <table style='border-collapse:collapse;font-size:0.85rem;color:#ddd;margin-bottom:8px'>
              <tr>
                <td style='padding:2px 12px'><b>Callsign</b></td><td style='padding:2px 12px'>{ep['callsign']}</td>
                <td style='padding:2px 12px'><b>Peak score</b></td>
                <td style='padding:2px 12px;color:{score_col}'><b>{ep['peak_score']:.3f}</b></td>
              </tr>
              <tr>
                <td style='padding:2px 12px'><b>Onset (UTC)</b></td><td style='padding:2px 12px'>{onset_utc}</td>
                <td style='padding:2px 12px'><b>Duration</b></td><td style='padding:2px 12px'>{ep['frame_count']} s</td>
              </tr>
              <tr>
                <td style='padding:2px 12px'><b>End (UTC)</b></td><td style='padding:2px 12px'>{end_utc}</td>
                <td style='padding:2px 12px'><b>Peak length</b></td><td style='padding:2px 12px'>{length_str}</td>
              </tr>
            </table>
            """
        else:
            meta_html = f"<p style='color:#888;font-size:0.8rem'>Metadata not found for {clip_path.name}</p>"

        video_html = f"""
        <video controls autoplay loop style='width:100%;max-width:960px;border-radius:6px'>
          <source src='{rel}' type='video/mp4'>
          Your browser does not support HTML5 video.
        </video>
        """

        with out:
            clear_output(wait=True)
            display(HTML(meta_html + video_html))

    def _update_clips(change):
        clips = discover_clips(date_dd.value)
        clip_dd.options = [(p.name, p) for p in clips]

    def _step(delta):
        if not clip_dd.options:
            return
        new_idx = max(0, min(len(clip_dd.options) - 1, clip_dd.index + delta))
        if new_idx != clip_dd.index:
            clip_dd.index = new_idx

    btn_prev.on_click(lambda _: _step(-1))
    btn_next.on_click(lambda _: _step(+1))
    date_dd.observe(_update_clips, names="value")
    clip_dd.observe(_show_clip, names="value")

    _update_clips(None)   # populate clip list for initial date

    display(widgets.VBox([
        widgets.HBox([date_dd, clip_dd, btn_prev, btn_next],
                     layout=widgets.Layout(gap="6px", align_items="center")),
        out,
    ]))